In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
import torch
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Dynamically set the dataset path relative to the script's location
dataset_path = "./dataset/fake_reviews_dataset.csv"

# Model save path and result directory are unchanged
model_save_path = os.path.join("./models/TinyLlama", "saved_tinyllama_model")
result_dir = os.path.join("./models/TinyLlama", "results")

# Create result directory if it doesn't exist
os.makedirs(result_dir, exist_ok=True)

# Load and preprocess the dataset
data = pd.read_csv(dataset_path)
data = data[['label', 'text']].fillna('')  # Ensure no missing values in text or labels

# Split into training, validation, and test sets
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    data['text'], data['label'], test_size=0.3, random_state=42
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42
)

# Convert datasets to Hugging Face's Dataset format
train_data = Dataset.from_dict({'text': train_texts.tolist(), 'label': train_labels.tolist()})
val_data = Dataset.from_dict({'text': val_texts.tolist(), 'label': val_labels.tolist()})
test_data = Dataset.from_dict({'text': test_texts.tolist(), 'label': test_labels.tolist()})

# Load TinyLlama tokenizer and model
model_name = "TinyLlama/TinyLlama_v1.1"  # Replace with the actual model identifier
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Add padding token if missing
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    model.resize_token_embeddings(len(tokenizer))

# Tokenize datasets
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding="max_length",
        truncation=True,
        max_length=64
    )

train_data = train_data.map(tokenize_function, batched=True).rename_column("label", "labels").remove_columns(["text"])
val_data = val_data.map(tokenize_function, batched=True).rename_column("label", "labels").remove_columns(["text"])
test_data = test_data.map(tokenize_function, batched=True).rename_column("label", "labels").remove_columns(["text"])

train_data.set_format("torch")
val_data.set_format("torch")
test_data.set_format("torch")

# Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Define training arguments
training_args = TrainingArguments(
    output_dir=result_dir,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=12,
    per_device_eval_batch_size=12,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,  # Enable mixed precision
    gradient_checkpointing=True,  # Reduce memory usage
)

# Define the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Train the model
trainer.train()

# Save the trained model and tokenizer
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)

# Evaluate on validation set
val_results = trainer.predict(val_data)
val_labels = val_data["labels"]
val_predictions = torch.argmax(torch.tensor(val_results.predictions), dim=1).numpy()

# Generate confusion matrix and classification report for validation
print("Validation Set Performance:")
print(classification_report(val_labels, val_predictions))
conf_matrix = confusion_matrix(val_labels, val_predictions)
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix (Validation)")
plt.savefig("val_confusion_matrix.png")
plt.show()

# Evaluate on test set
test_results = trainer.predict(test_data)
test_labels = test_data["labels"]
test_predictions = torch.argmax(torch.tensor(test_results.predictions), dim=1).numpy()

# Generate confusion matrix and classification report for test
print("Test Set Performance:")
print(classification_report(test_labels, test_predictions))
conf_matrix = confusion_matrix(test_labels, test_predictions)
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Greens")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix (Test)")
plt.savefig("test_confusion_matrix.png")
plt.show()
